In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# 02 — Component Ablation Study

Setiap variasi memakai data, seed, optimizer, jumlah topik, dan epoch
yang sama. Hanya komponen yang diuji yang diubah.


In [ ]:
from pathlib import Path
import sys

# Sesuaikan hanya apabila folder proyek berada di lokasi lain.
PROJECT_ROOT = Path("/content/drive/MyDrive/UDB/Riset/2026")
if not PROJECT_ROOT.exists():
    DRIVE_ROOT = Path("/content/drive/MyDrive/HMRNet_Experiments")
    if DRIVE_ROOT.exists():
        PROJECT_ROOT = DRIVE_ROOT

assert PROJECT_ROOT.exists(), (
    "Folder HMRNet_Experiments tidak ditemukan. "
    "Unggah/ekstrak proyek ke /content atau Google Drive."
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"
LOGS_DIR = RESULTS_DIR / "logs"
FEATURE_DIR = CHECKPOINTS_DIR / "features"

for directory in [TABLES_DIR, FIGURES_DIR, CHECKPOINTS_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
import numpy as np
import pandas as pd
from src.preprocessing import load_feature_bundle

bundle = load_feature_bundle(FEATURE_DIR)
raw_texts = bundle["raw_texts"]
processed_texts = bundle["processed_texts"]
bow = bundle["bow"]
bow_normalized = bundle["bow_normalized"]
tfidf = bundle["tfidf"]
semantic_embeddings = bundle["semantic_embeddings"]
document_embeddings = bundle["document_embeddings"]
vocabulary = bundle["vocabulary"]

print("Documents:", len(raw_texts))
print("Vocabulary:", len(vocabulary))
print("Semantic shape:", semantic_embeddings.shape)
print("Document shape:", document_embeddings.shape)


In [ ]:
import pandas as pd

from src.hmrnet import HMRNet, HMRNetConfig
from src.topic_metrics import evaluate_topics

N_TOPICS = 20
TOP_N_WORDS = 10
EPOCHS = 60


In [ ]:
ablations = {
    "HMRNet_Full": {},
    "HMRNet_-CRA": {
        "use_cra": False,
    },
    "HMRNet_-AF": {
        "fusion_mode": "average",
    },
    "HMRNet_-AL": {
        "use_alignment_loss": False,
    },
    "HMRNet_-DL": {
        "use_diversity_loss": False,
    },
    "HMRNet_-SL": {
        "use_stability_loss": False,
    },
    "HMRNet_Concat": {
        "fusion_mode": "concat",
    },
}


In [ ]:
%pip install -q --upgrade "gensim==4.4.0"

In [ ]:
rows = []
trained_models = {}

for name, overrides in ablations.items():
    print("\n", "=" * 72, "\n", name)
    config = HMRNetConfig(
        n_topics=N_TOPICS,
        epochs=EPOCHS,
        random_state=42,
        **overrides,
    )
    model = HMRNet(config).fit(
        bow=bow,
        semantic_embeddings=semantic_embeddings,
        document_embeddings=document_embeddings,
        vocabulary=vocabulary,
    )
    trained_models[name] = model
    model.save(CHECKPOINTS_DIR / f"ablation_{name}.pt")

    metrics = evaluate_topics(
        model_name=name,
        topics=model.get_topics(TOP_N_WORDS),
        processed_texts=processed_texts,
        top_n=TOP_N_WORDS,
    )
    weights = model.get_fusion_weights(
        bow, semantic_embeddings, document_embeddings
    ).mean(axis=0)
    metrics.update(
        {
            "Lexical_Weight": weights[0],
            "Semantic_Weight": weights[1],
            "Document_Weight": weights[2],
        }
    )
    rows.append(metrics)


In [ ]:
ablation_results = pd.DataFrame(rows)
full_cv = float(
    ablation_results.loc[
        ablation_results["Model"] == "HMRNet_Full", "C_v"
    ].iloc[0]
)
ablation_results["Delta_C_v_vs_Full"] = (
    ablation_results["C_v"] - full_cv
)

ablation_results = ablation_results.sort_values(
    "C_v", ascending=False
).reset_index(drop=True)

ablation_results.to_csv(
    TABLES_DIR / "experiment_02_ablation.csv",
    index=False,
)
ablation_results.to_excel(
    TABLES_DIR / "experiment_02_ablation.xlsx",
    index=False,
)
display(ablation_results.round(4))


In [ ]:
import matplotlib.pyplot as plt

plot_frame = ablation_results.sort_values("C_v")
plt.figure(figsize=(9, 5))
plt.barh(plot_frame["Model"], plot_frame["C_v"])
plt.xlabel("C_v coherence")
plt.title("HMRNet Component Ablation")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "experiment_02_ablation_cv.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/UDB/Riset/2026")
TABLES_DIR = PROJECT_ROOT / "results/tables"
CHECKPOINTS_DIR = PROJECT_ROOT / "results/checkpoints"

required_files = [
    TABLES_DIR / "experiment_02_ablation.csv",
    TABLES_DIR / "experiment_02_ablation.xlsx",
    CHECKPOINTS_DIR / "ablation_HMRNet_Full.pt",
    CHECKPOINTS_DIR / "ablation_HMRNet_-CRA.pt",
    CHECKPOINTS_DIR / "ablation_HMRNet_-AF.pt",
    CHECKPOINTS_DIR / "ablation_HMRNet_-AL.pt",
    CHECKPOINTS_DIR / "ablation_HMRNet_-DL.pt",
    CHECKPOINTS_DIR / "ablation_HMRNet_-SL.pt",
    CHECKPOINTS_DIR / "ablation_HMRNet_Concat.pt",
]

for file_path in required_files:
    print(f"{file_path.name}: {file_path.exists()}")